## This code creates a chatbot that answers questions about company employee data stored in an Excel sheet using RAG.

# Steps:

Install & Import: Loads necessary libraries for data processing, embeddings, and AI.


Process Data: Reads the Excel file, combines each row into a text chunk, and splits it for processing.

Create Knowledge Base: Generates vector embeddings for each text chunk and stores them in a local ChromaDB database.

Build QA Chain: Creates a retrieval system that finds relevant employee data and feeds it to an LLM to generate answers.

Query Loop: Takes user questions, retrieves the relevant data, generates an answer, and displays it with the source information.



In [1]:
# Install required packages
!pip install -qU pandas chromadb sentence-transformers

import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np
from datetime import datetime, timedelta

# Initialize model
model = SentenceTransformer('all-MiniLM-L6-v2')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

This code will generate an Excel file called employee_data.xlsx and has 50 employee records

In [2]:
# Generate sample employee data
np.random.seed(42)
num_employees = 50

first_names = ['James', 'Mary', 'John', 'Patricia', 'Robert', 'Jennifer', 'Michael', 'Linda',
               'William', 'Elizabeth', 'David', 'Barbara', 'Richard', 'Susan', 'Joseph', 'Jessica',
               'Thomas', 'Sarah', 'Charles', 'Karen', 'Christopher', 'Nancy', 'Daniel', 'Lisa',
               'Matthew', 'Betty', 'Anthony', 'Margaret', 'Mark', 'Sandra', 'Donald', 'Ashley',
               'Steven', 'Kimberly', 'Paul', 'Emily', 'Andrew', 'Donna', 'Joshua', 'Michelle']

last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis',
              'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson',
              'Thomas', 'Taylor', 'Moore', 'Jackson', 'Martin', 'Lee', 'Perez', 'Thompson',
              'White', 'Harris', 'Sanchez', 'Clark', 'Ramirez', 'Lewis', 'Robinson', 'Walker',
              'Young', 'Allen', 'King', 'Wright', 'Scott', 'Torres', 'Nguyen', 'Hill', 'Flores']

data = {
    'employee_id': [f'E{1000+i}' for i in range(num_employees)],
    'first_name': np.random.choice(first_names, num_employees),
    'last_name': np.random.choice(last_names, num_employees),
    'age': np.random.randint(22, 65, num_employees),
    'gender': np.random.choice(['Male', 'Female'], num_employees),
    'birth_date': [datetime(1990, 1, 1) + timedelta(days=np.random.randint(0, 365*30)) for _ in range(num_employees)],
    'days_attended': np.random.randint(200, 250, num_employees),
    'days_missed': np.random.randint(0, 20, num_employees),
    'job_title': np.random.choice(['Software Engineer', 'Data Analyst', 'Project Manager',
                                   'HR Specialist', 'Sales Executive', 'Marketing Manager',
                                   'Accountant', 'DevOps Engineer', 'Product Designer'], num_employees),
    'salary': np.random.randint(50000, 120000, num_employees)
}

df = pd.DataFrame(data)
df['birth_date'] = df['birth_date'].dt.strftime('%Y-%m-%d')
df.to_excel('employee_data.xlsx', index=False)

# Create knowledge base
df['text'] = df.apply(lambda row: f"Name: {row['first_name']} {row['last_name']}, Age: {row['age']}, Gender: {row['gender']}, Job: {row['job_title']}, Salary: ${row['salary']}, Days: {row['days_attended']} attended, {row['days_missed']} missed", axis=1)

In [ ]:
# Initialize ChromaDB
client = chromadb.Client()
try:
    collection = client.get_collection("employees")
except:
    collection = client.create_collection("employees")
    embeddings = model.encode(df['text'].tolist()).tolist()
    collection.add(
        ids=[str(i) for i in range(len(df))],
        embeddings=embeddings,
        documents=df['text'].tolist()
    )

# Smart chatbot function with natural language understanding
def smart_chatbot(question):
    question_lower = question.lower()

    # For counting questions, search the ENTIRE dataset, not just top results
    if any(word in question_lower for word in ['how many', 'count', 'number']):
        if 'male' in question_lower:
            male_count = df[df['gender'] == 'Male'].shape[0]
            return f"There are {male_count} male employees in the company."
        elif 'woman' in question_lower:
            female_count = df[df['gender'] == 'Female'].shape[0]
            return f"There are {female_count} female employees in the company."
        elif 'employee' in question_lower:
            return f"There are {len(df)} total employees in the company."

    # For salary questions, use the ENTIRE dataset for accurate averages
    if any(word in question_lower for word in ['average salary', 'average pay', 'avg salary']):
        avg_salary = df['salary'].mean()
        return f"The average salary for all employees is ${avg_salary:,.2f}"

    # For other questions, use the vector search
    query_embedding = model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=5)
    context = results['documents'][0]

    # Salary questions for specific groups
    if any(word in question_lower for word in ['salary', 'pay', 'earn']):
        if 'average' in question_lower:
            salaries = []
            for doc in context:
                if 'Salary: $' in doc:
                    try:
                        salary_str = doc.split('Salary: $')[1].split(',')[0]
                        salaries.append(float(salary_str))
                    except:
                        continue
            if salaries:
                avg = sum(salaries) / len(salaries)
                return f"Average salary of matching employees: ${avg:,.2f}"
        elif 'highest' in question_lower:
            max_salary = 0
            max_employee = ""
            for doc in context:
                if 'Salary: $' in doc:
                    try:
                        salary_str = doc.split('Salary: $')[1].split(',')[0]
                        salary = float(salary_str)
                        if salary > max_salary:
                            max_salary = salary
                            max_employee = doc.split(',')[0]
                    except:
                        continue
            if max_employee:
                return f"Highest paid among matches: {max_employee} with ${max_salary:,.2f}"

    # Job title questions
    elif any(word in question_lower for word in ['job', 'role', 'position', 'title']):
        jobs = {}
        for doc in context:
            if 'Job: ' in doc:
                job = doc.split('Job: ')[1].split(',')[0]
                jobs[job] = jobs.get(job, 0) + 1
        if jobs:
            response = "Job titles found in matches:\n"
            for job, count in jobs.items():
                response += f"- {job}: {count} employees\n"
            return response

    # General list questions
    elif any(word in question_lower for word in ['list', 'show', 'who', 'what']):
        response = "Matching employees:\n"
        for i, doc in enumerate(context, 1):
            response += f"{i}. {doc}\n"
        return response

    # Default response - show what was found
    return f"I found these employees relevant to your question:\n" + "\n".join([f"• {doc}" for doc in context])

# Chatbot loop
print("🤖 Employee Chatbot Ready!")
print("Ask anything about employees - type 'exit' to quit\n")

while True:
    query = input("You: ")
    if query.lower() in ['exit', 'quit']:
        break

    response = smart_chatbot(query)
    print(f"Bot: {response}\n")

🤖 Employee Chatbot Ready!
Ask anything about employees - type 'exit' to quit

